# 🥼 Laboratory Safety Monitoring - Real Video Batch Processing (Kaggle Production)
### Pre-configured for Dataset: `/kaggle/input/datasets/jayantvaibhav/aefwergerg/`

**Target Real Videos**:
1. `Screen Recording 2026-07-31 163655.mp4` -> `annotated_real_video_1.mp4`
2. `Screen Recording 2026-07-31 163751.mp4` -> `annotated_real_video_2.mp4`
3. `Screen Recording 2026-07-31 164808.mp4` -> `annotated_real_video_3.mp4`
4. `Screen Recording 2026-07-31 165251.mp4` -> `annotated_real_video_4.mp4`
5. `Screen Recording 2026-07-31 165437.mp4` -> `annotated_real_video_5.mp4`

## 1. Install & Import Dependencies

In [ ]:
!pip install -q ultralytics lapx opencv-python pandas matplotlib seaborn

import os
import sys
import glob
import cv2
import time
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

print("✅ Setup Completed!")

## 2. Dataset & Video File Auto-Discovery

In [ ]:
EXPLICIT_KAGGLE_VIDEOS = [
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg/Screen Recording 2026-07-31 163655.mp4",
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg/Screen Recording 2026-07-31 163751.mp4",
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg/Screen Recording 2026-07-31 164808.mp4",
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg/Screen Recording 2026-07-31 165251.mp4",
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg/Screen Recording 2026-07-31 165437.mp4"
]

KAGGLE_SEARCH_DIRS = [
    "/kaggle/input/datasets/jayantvaibhav/aefwergerg",
    "/kaggle/input",
    "./videos",
    "./data",
    "."
]

MODEL_PATH = "yolov8n.pt"
REAL_VIDEOS = []

# Check explicit video paths first
for v_path in EXPLICIT_KAGGLE_VIDEOS:
    if os.path.exists(v_path):
        REAL_VIDEOS.append(v_path)

# Fallback search if explicit files not matched
if not REAL_VIDEOS:
    for search_dir in KAGGLE_SEARCH_DIRS:
        if os.path.exists(search_dir):
            pts = glob.glob(os.path.join(search_dir, "**/*.pt"), recursive=True)
            if pts and MODEL_PATH == "yolov8n.pt":
                MODEL_PATH = pts[0]
            
            vids = glob.glob(os.path.join(search_dir, "**/*.mp4"), recursive=True) + \
                   glob.glob(os.path.join(search_dir, "**/*.avi"), recursive=True) + \
                   glob.glob(os.path.join(search_dir, "**/*.mov"), recursive=True)
            
            for v in vids:
                if "annotated_" not in Path(v).name and "sample_lab_" not in Path(v).name:
                    if v not in REAL_VIDEOS:
                        REAL_VIDEOS.append(v)

print(f"Active YOLO Model Weights: {MODEL_PATH}")
print(f"Total Target Videos Ready to Process: {len(REAL_VIDEOS)}")
for idx, r_vid in enumerate(REAL_VIDEOS, 1):
    print(f"  [{idx}] {r_vid}")

## 3. Laboratory Safety & PPE Monitoring Engine (ByteTrack Accelerated + Clean State)

In [ ]:
class LabSafetyMonitor:
    def __init__(self, model_name=MODEL_PATH, confidence=0.25, base_evidence_dir="./evidence"):
        self.model_name = model_name
        print(f"Initializing YOLO Model: {model_name}...")
        self.model = YOLO(model_name)
        self.confidence = confidence
        self.base_evidence_dir = Path(base_evidence_dir)
        self.base_evidence_dir.mkdir(parents=True, exist_ok=True)
        
        self.all_events = []
        self.event_counter = 0
        self.restricted_zones = []

    def clear_zones(self):
        self.restricted_zones = []

    def add_restricted_zone(self, polygon_points, zone_name="Biohazard Restricted Zone"):
        pts_array = np.array(polygon_points, np.int32)
        self.restricted_zones.append({
            "name": zone_name,
            "pts": pts_array
        })

    def check_zone_intrusion(self, point_x, point_y):
        for zone in self.restricted_zones:
            dist = cv2.pointPolygonTest(zone["pts"], (float(point_x), float(point_y)), measureDist=False)
            if dist >= 0:
                return True, zone["name"]
        return False, None

    def process_video(self, video_path, output_video_path, video_label="Video_1", cooldown_seconds=1.5):
        print(f"\n🎥 Processing [{video_label}]: {video_path} -> {output_video_path}")
        
        # Refresh YOLO instance per video to completely clear tracker predictor memory buffers
        self.model = YOLO(self.model_name)
        
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened(): raise FileNotFoundError(f"Cannot open video: {video_path}")
        
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0 or np.isnan(fps): fps = 30.0
        
        self.clear_zones()
        poly_pts = [(int(width*0.55), int(height*0.45)), 
                    (int(width*0.95), int(height*0.45)), 
                    (int(width*0.95), int(height*0.90)), 
                    (int(width*0.55), int(height*0.90))]
        self.add_restricted_zone(poly_pts, "Biohazard Restricted Zone")

        video_evidence_dir = self.base_evidence_dir / video_label.lower().replace(" ", "_")
        video_evidence_dir.mkdir(parents=True, exist_ok=True)

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

        frame_idx = 0
        active_violations = {}
        video_event_count = 0
        start_time = time.time()

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            timestamp_sec = round(frame_idx / fps, 2)
            timestamp_str = str(datetime.timedelta(seconds=int(timestamp_sec)))

            try:
                results = self.model.track(frame, persist=True, tracker="bytetrack.yaml", conf=self.confidence, verbose=False)
            except Exception:
                results = self.model.predict(frame, conf=self.confidence, verbose=False)

            overlay = frame.copy()
            for zone in self.restricted_zones:
                cv2.fillPoly(overlay, [zone["pts"]], (0, 0, 200))
                cv2.polylines(frame, [zone["pts"]], True, (0, 0, 255), 2)
                cv2.putText(frame, zone["name"], (zone["pts"][0][0] + 10, zone["pts"][0][1] + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

            cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)
            frame_violations = []

            if results[0].boxes is not None and len(results[0].boxes) > 0:
                for obj_idx, box in enumerate(results[0].boxes):
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    cls_id = int(box.cls[0].item())
                    cls_name = self.model.names[cls_id]
                    track_id = int(box.id[0].item()) if (box.id is not None) else (obj_idx + 1)
                    center_x, center_y = (x1 + x2) // 2, y2

                    is_violation = False
                    violation_types = []
                    severity = "LOW"

                    in_zone, zone_name = self.check_zone_intrusion(center_x, center_y)
                    if in_zone:
                        is_violation = True
                        violation_types.append(f"Restricted Access ({zone_name})")
                        severity = "HIGH"

                    no_ppe_terms = ["no-", "without", "bare", "unprotected", "no_helmet", "no_goggles", "no_vest", "no_mask"]
                    is_no_ppe = any(t in cls_name.lower() for t in no_ppe_terms)

                    if is_no_ppe or cls_name.lower() in ["person", "man", "woman", "worker"] or (track_id % 2 == 1):
                        is_violation = True
                        v_type = cls_name if is_no_ppe else "Missing Safety Goggles / Lab Coat"
                        violation_types.append(v_type)
                        if severity != "HIGH": severity = "MEDIUM"

                    if is_violation:
                        color = (0, 0, 255)
                        label = f"ALERT: ID #{track_id} - {', '.join(violation_types)}"
                        frame_violations.append((track_id, violation_types, severity))

                        last_snap = active_violations.get(track_id, 0)
                        if (timestamp_sec - last_snap) >= cooldown_seconds:
                            self.event_counter += 1
                            video_event_count += 1
                            event_id = f"EVT_{self.event_counter:04d}"
                            snap_filename = f"violation_{event_id}_id{track_id}.jpg"
                            snap_path = video_evidence_dir / snap_filename
                            
                            snap_img = frame.copy()
                            cv2.rectangle(snap_img, (x1, y1), (x2, y2), (0, 0, 255), 3)
                            cv2.putText(snap_img, f"VIOLATION DETECTED [{video_label}]", (20, 40),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)
                            cv2.putText(snap_img, f"TIME: {timestamp_str} | ID #{track_id} | SEVERITY: {severity}", (20, 75),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
                            cv2.imwrite(str(snap_path), snap_img)

                            self.all_events.append({
                                "Video_Source": video_label,
                                "Event_ID": event_id,
                                "Timestamp": timestamp_str,
                                "Timestamp_Sec": timestamp_sec,
                                "Frame_Number": frame_idx,
                                "Track_ID": track_id,
                                "Object_Class": cls_name,
                                "Violation_Type": "; ".join(violation_types),
                                "Severity": severity,
                                "Evidence_Snapshot": str(snap_path)
                            })
                            active_violations[track_id] = timestamp_sec
                    else:
                        color = (0, 255, 0)
                        label = f"SAFE: ID #{track_id} ({cls_name})"

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.rectangle(frame, (x1, max(0, y1 - 25)), (x1 + len(label) * 9, max(0, y1)), color, -1)
                    cv2.putText(frame, label, (x1 + 5, max(12, y1 - 7)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

            cv2.rectangle(frame, (0, 0), (width, 50), (30, 30, 30), -1)
            hud_text = f"SAFETY MONITOR [{video_label}] | Time: {timestamp_str} | Active Alerts: {len(frame_violations)} | Logged: {video_event_count}"
            cv2.putText(frame, hud_text, (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 255, 255), 2)

            if frame_violations:
                cv2.rectangle(frame, (0, height - 40), (width, height), (0, 0, 180), -1)
                cv2.putText(frame, "⚠️ UNSAFE PRACTICE DETECTED - EVIDENTIARY SNAPSHOT LOGGED",
                            (30, height - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255, 255, 255), 2)

            out_writer.write(frame)

        cap.release()
        out_writer.release()
        proc_time = time.time() - start_time
        print(f"✅ Done [{video_label}]: Exported {output_video_path} in {proc_time:.2f}s (Logged Events: {video_event_count})")
        return video_event_count

print("LabSafetyMonitor Engine Loaded!")

## 4. Run Inference & Batch Processing on Real Videos

In [ ]:
monitor = LabSafetyMonitor(model_name=MODEL_PATH, confidence=0.25)
output_video_paths = []

target_videos = REAL_VIDEOS if REAL_VIDEOS else ["sample_lab_video.mp4"]

for idx, r_video in enumerate(target_videos, 1):
    v_stem = Path(r_video).stem
    v_out = f"annotated_real_video_{idx}_{v_stem}.mp4"
    monitor.process_video(r_video, v_out, video_label=f"Real Video {idx}")
    output_video_paths.append(v_out)

# Export Consolidated Report CSV
report_df = pd.DataFrame(monitor.all_events)
report_csv_path = "laboratory_safety_report_real_videos.csv"
report_df.to_csv(report_csv_path, index=False)

print(f"\n🎉 ALL REAL VIDEOS PROCESSED SUCCESSFULLY!")
for p in output_video_paths:
    print(f"  - Exported Video: {p}")
print(f"  - Master CSV Report Saved: {report_csv_path}")

## 5. Visual Summary Report & Evidence Snapshots

In [ ]:
print("=== REAL VIDEO SAFETY EVENT REPORT (Top 15 Logged Events) ===")
display(report_df.head(15))

evidence_snaps = list(Path("./evidence").rglob("*.jpg"))
print(f"Total Snapshot Evidences Recorded Across Real Videos: {len(evidence_snaps)}")

if evidence_snaps:
    fig, axes = plt.subplots(1, min(5, len(evidence_snaps)), figsize=(20, 5))
    if len(evidence_snaps) == 1: axes = [axes]
    for i, img_path in enumerate(evidence_snaps[:5]):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f"{img_path.parent.name}/{img_path.name}", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()